# Aula 10 — Filtrando e cortando mensagens

**Objetivo:** impedir que a conversa longa estoure tokens, latência e janela
do modelo. Ao final você deve saber quando APAGAR do estado, quando FILTRAR
só na chamada e quando CORTAR por orçamento — e o preço de cada escolha.

**Pré-requisitos:** Aula 06 (histórico cresce a cada turno) e Aula 08
(`RemoveMessage`, `add_messages`).

**Como usar:** rode em ordem. Autocontido, 100% offline (sem LLM, sem chave).
Comece pela CÉLULA 0.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph tiktoken
print('dependências prontas.')

## 1. Analogia: a comanda que não cabe no bolso

A comanda da mesa 12 tem 40 itens. O cozinheiro precisa ver tudo; o caixa só
precisa do total; e ninguém consegue carregar 40 linhas no bolso. Três saídas:

| Comanda | No LangGraph | Preço |
|---|---|---|
| rasgar itens antigos | **apagar do estado** (`RemoveMessage`) | some para sempre (próximos turnos também) |
| mostrar só o total ao caixa | **filtrar só no `invoke`** (`messages[-1:]`) | estado intacto, mas paga o histórico inteiro no checkpoint |
| resumir em 10 linhas | **cortar por tokens** (`trim_messages`) | automático por orçamento, mas pode decepar contexto |

Regra de bolso: dado sensível/antigo sem volta → apaga; contexto que o modelo
não precisa *nesta* chamada → filtra; conversa longa em produção → corta.

In [ ]:
# CÉLULA 1 — a conversa de exemplo + contador de tokens (100% offline)
import tiktoken
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

_ENC = tiktoken.get_encoding('cl100k_base')

def conta_tokens(msgs) -> int:
    # aproximação didática; produção usa o contador do próprio modelo
    return sum(3 + len(_ENC.encode(m.content)) for m in msgs)

conversa = [
    SystemMessage(content='Você é um atendente breve e educado da lanchonete.', id='s0'),
    HumanMessage(content='quero um pastel de carne e um suco de laranja bem gelado', id='m1'),
    AIMessage(content='Anotado: pastel e suco. Mais alguma coisa?', id='m2'),
    HumanMessage(content='adiciona um refri e me fala o total da conta por favor', id='m3'),
    AIMessage(content='Total R$ 14. Posso fechar a conta?', id='m4'),
]
print('mensagens:', len(conversa), '| tokens:', conta_tokens(conversa))
print('-> sem poda, cada turno reenvia tudo isso ao modelo.')

## 2. Conceito: onde a poda acontece (estado vs chamada)

- **Apagar do estado**: o `RemoveMessage` entra no update e o reducer remove
  de verdade. Próximos turnos, checkpoints e `get_state_history` **não** veem
  mais. Irreversível (a menos que você tenha log fora).
- **Filtrar na chamada**: `llm.invoke(state['messages'][-1:])`. O estado segue
  completo; só *esta* chamada vê o recorte. Barato de implementar, mas o
  histórico continua custando checkpoint e `get_state`.
- **Cortar por orçamento**: `trim_messages(msgs, max_tokens=N, ...)` devolve
  lista podada. Automático, mas cego: corta por tamanho, não por importância
  (pode decepar justo o pedido original).

In [ ]:
# CÉLULA 2 — apagar do estado: ficam as 2 últimas (rode e confira)
from langchain_core.messages import RemoveMessage
from langgraph.graph import START, END, MessagesState, StateGraph

def filtra(state: MessagesState) -> dict:
    apaga = [RemoveMessage(id=m.id) for m in state['messages'][:-2]]
    return {'messages': apaga}

b = StateGraph(MessagesState)
b.add_node('filtra', filtra)
b.add_edge(START, 'filtra'); b.add_edge('filtra', END)
out = b.compile().invoke({'messages': conversa})
print([(type(m).__name__, m.id) for m in out['messages']])
assert [m.id for m in out['messages']] == ['m3', 'm4']
print('OK — s0..m2 sumiram do ESTADO (não só da chamada).')

## 3. Dissecando o `RemoveMessage` (o que ele é e o que não é)

- `RemoveMessage(id='m1')` **não é mensagem de conversa**: é uma ordem de
  remoção que só faz sentido dentro de um update com `add_messages`.
- O reducer casa pelo **`id`**: cada `RemoveMessage` apaga a mensagem de
  mesmo `id`. Sem `id` nas mensagens, não há como apagar seletivamente —
  por isso mensagens com `id` explícito são boa prática em produção.
- Sem reducer (`messages: list` puro), o `RemoveMessage` **não funciona**:
  overwrite não sabe interpretar a ordem (Aula 08).

In [ ]:
# CÉLULA 3 — filtrar só no invoke: estado intacto, visão recortada
def eco(state: MessagesState) -> dict:
    vista = state['messages'][-1:]          # o LLM veria só isto
    return {'messages': [AIMessage(content=f'vi {len(vista)} msg(s): {vista[0].content}')]}

b2 = StateGraph(MessagesState)
b2.add_node('eco', eco)
b2.add_edge(START, 'eco'); b2.add_edge('eco', END)
out2 = b2.compile().invoke({'messages': conversa})
print('estado final tem', len(out2['messages']), 'msgs (5 originais + 1 resposta)')
print('resposta:', repr(out2['messages'][-1].content))
assert len(out2['messages']) == 6
print('OK — nada foi apagado; o recorte existiu só dentro do nó.')

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. `trim_messages`: orçamento + system preservado

- `max_tokens=40, strategy='last'`: anda do fim para o começo, acumulando até
  o teto. `include_system=True` reserva o `SystemMessage` fora da conta do corte.
- `token_counter` aqui é nossa função `conta_tokens` (transparente e offline).
  Produção usa o contador do próprio modelo ou `'approximate'` (rápido).

In [ ]:
# CÉLULA 4 — trim com system preservado (rode e confira o teto)
from langchain_core.messages import trim_messages

cortada = trim_messages(conversa, max_tokens=40, strategy='last',
                        token_counter=conta_tokens,
                        include_system=True, allow_partial=False)
print([(type(m).__name__, m.id) for m in cortada])
print('tokens:', conta_tokens(cortada))
assert conta_tokens(cortada) <= 40
assert cortada[0].id == 's0' and cortada[-1].id == 'm4'
print('OK — teto respeitado, system na frente, fim preservado.')

### 4b. Sem `include_system`, o SYS cai — e `start_on` pode comer tudo

- `include_system=False`: o `SystemMessage` entra na conta do corte como
  mensagem comum (aqui, some). Sem a instrução fixa, o modelo perde o 'manual'.
- `start_on='human'`: exige que o recorte comece em fala humana (a maioria dos
  modelos espera isso). Com orçamento apertado, ele descarta até sobrar quase
  nada — aqui, só o system. Orçamento e `start_on` brigam: dimensione com folga.

In [ ]:
# CÉLULA 5 — sem system e start_on faminto (rode e compare)
from langchain_core.messages import trim_messages

sem_sys = trim_messages(conversa, max_tokens=40, strategy='last',
                        token_counter=conta_tokens,
                        include_system=False, allow_partial=False)
print('sem system:', [(type(m).__name__, m.id) for m in sem_sys])
assert [m.id for m in sem_sys] == ['m3', 'm4']

faminto = trim_messages(conversa, max_tokens=40, strategy='last',
                        token_counter=conta_tokens,
                        include_system=True, start_on='human', allow_partial=False)
print('start_on=human:', [(type(m).__name__, m.id) for m in faminto])
print('OK — sem system o SYS cai; start_on apertado come o histórico.')

### 4c. `allow_partial`: aproveitar o fim da mensagem longa

- `False`: mensagem que não cabe **cai inteira** (mesmo que só o começo
  estoure). Seguro, mas desperdiça.
- `True`: inclui o **fim** da mensagem longa (strategy `last`), cortada por
  linhas. Útil quando o fim carrega a informação (ex: último parágrafo).
- Detalhe: a divisão é por **linhas de texto**, não por token exato — mensagem
  de uma linha só e longa demais cai inteira mesmo com `True`.

In [ ]:
# CÉLULA 6 — parcial aproveita o fim (rode e confira)
from langchain_core.messages import trim_messages

longas = [HumanMessage(
    content='linha um do pedido\nlinha dois do pedido\nlinha três do pedido\nlinha quatro do pedido',
    id='m1'),
    AIMessage(content='Anotado.', id='m2'),
    HumanMessage(content='adiciona um refri', id='m3')]

inteira = trim_messages(longas, max_tokens=30, strategy='last',
                        token_counter=conta_tokens,
                        include_system=False, allow_partial=False)
parcial = trim_messages(longas, max_tokens=30, strategy='last',
                        token_counter=conta_tokens,
                        include_system=False, allow_partial=True)
print('partial=False:', [(m.id, m.content) for m in inteira])
print('partial=True :', [(m.id, m.content) for m in parcial])
assert [m.id for m in inteira] == ['m2', 'm3']
assert parcial[0].id == 'm1' and 'quatro' in parcial[0].content
print('OK — False derruba a longa inteira; True salva o fim dela.')

## 5. Erros clássicos

1. **Apagar sem `id` nas mensagens.** `RemoveMessage` casa por `id`; histórico
   sem ids explícitos não permite poda seletiva. Dê `id` desde o começo.
2. **Achar que filtrar no `invoke` economiza checkpoint.** Economiza tokens da
   chamada, mas o estado continua crescendo (sessão, `get_state`, disco).
3. **Cortar sem `include_system`.** O modelo perde o 'manual' e muda de
   comportamento no meio da conversa — sintoma clássico e silencioso.
4. **`start_on` com orçamento apertado.** Ele descarta até satisfazer a regra
   e pode sobrar quase nada (CÉLULA 5). Dimensione com folga.
5. **Contador errado.** `token_counter=len` conta **mensagens**, não tokens
   (5 msgs = '5 tokens'). Use o contador do modelo ou tiktoken aproximado.
6. **Podar o pedido original.** Trim cego pode decepar justo o contexto que o
   turno atual referencia ('isso' da Aula 06). Podar pede critério, não só teto.

## 6. Exercício guiado — comanda com teto de 40 tokens

Situação: a mesa 12 tem a conversa de 5 mensagens da CÉLULA 1. O caixa precisa
enxergar no máximo 40 tokens, **sem perder** a instrução do sistema nem o
fechamento (`m4`). Aplique `trim_messages` e confira as 3 condições.

In [ ]:
# CÉLULA 7 — exercício guiado (teto + system + fim)
from langchain_core.messages import trim_messages

caixa = trim_messages(conversa, max_tokens=40, strategy='last',
                      token_counter=conta_tokens,
                      include_system=True, allow_partial=False)
print([(type(m).__name__, m.id) for m in caixa], '->', conta_tokens(caixa), 'tokens')
assert conta_tokens(caixa) <= 40, 'estourou o teto!'
assert caixa[0].id == 's0', 'perdeu o manual!'
assert caixa[-1].id == 'm4', 'perdeu o fechamento!'
print('OK — teto respeitado, manual na frente, fechamento no fim.')

## 7. Desafio — poda que preserva o pedido original

O trim da CÉLULA 4 manteve `m4` mas pode ter derrubado `m1` (o pedido original
que o 'isso' referencia). Projete uma poda melhor e valide:

1. Garanta que `m1` (pedido original) **sempre** sobreviva, mesmo com teto 40.
   Dica: `filter_messages` por `id` + concatenação manual, ou `start_on` com
   folga maior — compare as duas saídas.
2. Meça os tokens das duas abordagens com `conta_tokens`.
3. **Pegadinha:** qual das duas quebra se `m1` for gigante? (Resposta: a manual
   estoura o teto; a automática com `allow_partial=True` salvaria o fim dela.
   Combine as duas: âncora manual + trim no meio.)

## 8. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] Apagar do estado vs filtrar na chamada: o que cada um custa e preserva?
- [ ] Por que `RemoveMessage` precisa de `id` — e de reducer?
- [ ] O que `include_system`, `start_on` e `allow_partial` mudam no corte?
- [ ] Por que `token_counter=len` mente — e o que usar no lugar?
- [ ] Quando o trim decepra o contexto que o turno atual precisa?
- [ ] Na sua comanda real, o que seria apagável, filtrável e intocável?

**Próxima aula (11 — Streaming, interrupção e HITL):** ver o grafo executando
(`updates`/`values`/`messages`), pausar para aprovação humana e retomar — a
base de todo humano-no-loop do bloco.